<a href="https://colab.research.google.com/github/ahmed8809/FlyRank_intern/blob/main/work/notebooks/w04_baseline_score.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-07 — Baseline Action Score and Top-20 Review

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [9]:
import duckdb
from google.colab import userdata

HF_TOKEN = userdata.get("HF_TOKEN")

con = duckdb.connect()

con.execute(
    f"CREATE SECRET (TYPE huggingface, TOKEN {HF_TOKEN})"
)

rel = "hf://datasets/FlyRank/internship-warehouse"

print("Connection ready")

Connection ready


In [10]:
dim_schema = con.sql(f"""
DESCRIBE
SELECT *
FROM read_parquet(
    '{rel}/dim_content.parquet'
)
""").df()

display(dim_schema)

,column_name,column_type,null,key,default,extra
0,client_hash_id,VARCHAR,YES,None,None,None
1,content_hash_id,VARCHAR,YES,None,None,None
2,keyword_hash_id,VARCHAR,YES,None,None,None
3,url_hash_id,VARCHAR,YES,None,None,None
4,keyword_char_count,BIGINT,YES,None,None,None
5,keyword_token_count,BIGINT,YES,None,None,None
6,url_char_count,BIGINT,YES,None,None,None
7,content_created_date,DATE,YES,None,None,None
8,content_updated_date,DATE,YES,None,None,None
9,content_type,VARCHAR,YES,None,None,None


In [14]:
staleness_buckets = con.sql(f"""
WITH page_data AS (
    SELECT
        f.client_hash_id,
        f.content_hash_id,
        f.report_date,
        d.content_updated_date,

        DATE_DIFF(
            'day',
            d.content_updated_date,
            f.report_date
        ) AS stale_days

    FROM read_parquet(
        '{rel}/fact_content_daily_performance/month=2026-03/*.parquet'
    ) f

    INNER JOIN read_parquet(
        '{rel}/dim_content.parquet'
    ) d
        ON f.client_hash_id = d.client_hash_id
        AND f.content_hash_id = d.content_hash_id

    WHERE
        f.gsc_data_available IS TRUE
        AND d.content_updated_date IS NOT NULL
        AND d.content_updated_date <= f.report_date
)

SELECT
    CASE
        WHEN stale_days < 90 THEN 'fresh_<90d'
        WHEN stale_days < 180 THEN 'stale_90_180d'
        WHEN stale_days < 365 THEN 'stale_180_365d'
        ELSE 'very_stale_365d+'
    END AS staleness_bucket,

    COUNT(DISTINCT content_hash_id) AS n

FROM page_data

GROUP BY 1
ORDER BY 1;
""").df()

display(staleness_buckets)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,staleness_bucket,n
0,fresh_<90d,26479
1,stale_180_365d,213
2,stale_90_180d,1398


In [15]:
volume_buckets = con.sql(f"""
WITH page_volume AS (
    SELECT
        client_hash_id,
        content_hash_id,
        SUM(COALESCE(gsc_impressions, 0)) AS impressions

    FROM read_parquet(
        '{rel}/fact_content_daily_performance/month=2026-03/*.parquet'
    )

    WHERE gsc_data_available IS TRUE

    GROUP BY
        client_hash_id,
        content_hash_id
)

SELECT
    CASE
        WHEN impressions = 0 THEN 'zero'
        WHEN impressions < 100 THEN 'low_<100'
        WHEN impressions < 1000 THEN 'medium_100_999'
        WHEN impressions < 10000 THEN 'high_1k_9.9k'
        ELSE 'very_high_10k+'
    END AS volume_bucket,

    COUNT(*) AS n

FROM page_volume

GROUP BY 1
ORDER BY 1;
""").df()

display(volume_buckets)

,volume_bucket,n
0,high_1k_9.9k,39181
1,low_<100,75297
2,medium_100_999,56383
3,very_high_10k+,5877


### Signal 1 — Staleness

**Verdict: CONFIRMED**

The March data contains 1,611 content items that were at least 90 days stale
(1,398 between 90–180 days and 213 between 180–365 days). This gives the
signal meaningful variation rather than treating all pages as equally fresh.

Staleness is also directly connected to the refresh-flag logic discussed in
the FlyRank session.

### Signal 2 — Search volume

**Verdict: CONFIRMED**

Search impressions vary substantially across the content inventory, from fewer
than 100 impressions to more than 10,000. This gives the rule a useful way to
distinguish pages with little observed demand from pages with meaningful search
visibility.

Search volume is also connected to the quick-win logic discussed in the
FlyRank session.

## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

### Baseline rule

I prioritize content for refresh when two observed signals are both present:

1. The content is at least 90 days stale.
2. The content received at least 1,000 Google Search Console impressions during March 2026.

The score is the March GSC impression count for pages satisfying both
conditions. Pages that do not satisfy both conditions are not prioritized.

**Reason code:** `stale_visible`

**Action:** `refresh`

The rule is intentionally simple and hand-written. It uses only information
observed within the March 2026 window and does not use future performance or
label-derived information.

## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

In [17]:
queue = con.sql(f"""
WITH page_metrics AS (

    SELECT
        f.client_hash_id,
        f.content_hash_id,

        SUM(COALESCE(f.gsc_impressions, 0)) AS gsc_impressions,

        MAX(f.report_date) AS report_date,

        d.content_updated_date

    FROM read_parquet(
        '{rel}/fact_content_daily_performance/month=2026-03/*.parquet'
    ) f

    INNER JOIN read_parquet(
        '{rel}/dim_content.parquet'
    ) d
        ON f.client_hash_id = d.client_hash_id
        AND f.content_hash_id = d.content_hash_id

    WHERE
        f.gsc_data_available IS TRUE

    GROUP BY
        f.client_hash_id,
        f.content_hash_id,
        d.content_updated_date
),

signals AS (

    SELECT
        *,
        DATE_DIFF(
            'day',
            content_updated_date,
            report_date
        ) AS stale_days

    FROM page_metrics

    WHERE
        content_updated_date IS NOT NULL
        AND content_updated_date <= report_date
),

scored AS (

    SELECT
        client_hash_id,
        content_hash_id,
        report_date,
        content_updated_date,
        stale_days,
        gsc_impressions,

        (
            CASE WHEN stale_days >= 90 THEN 1 ELSE 0 END
            *
            CASE WHEN gsc_impressions >= 1000 THEN 1 ELSE 0 END
            *
            gsc_impressions
        ) AS score,

        CASE
            WHEN stale_days >= 90
                 AND gsc_impressions >= 1000
            THEN 'stale_visible'
            ELSE 'not_stale_visible'
        END AS reason_code,

        CASE
            WHEN stale_days >= 90
                 AND gsc_impressions >= 1000
            THEN 'refresh'
            ELSE 'monitor'
        END AS action

    FROM signals
)

SELECT
    *,
    ROW_NUMBER() OVER (
        ORDER BY
            score DESC,
            stale_days DESC,
            gsc_impressions DESC
    ) AS rank

FROM scored

ORDER BY rank;
""").df()

display(queue.head(20))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,client_hash_id,content_hash_id,report_date,content_updated_date,stale_days,gsc_impressions,score,reason_code,action,rank
0,client_20259bd6705d81d4,content_097459d155cccb26,2026-03-31,2025-11-27,124,37930.0,37930.0,stale_visible,refresh,1
1,client_20259bd6705d81d4,content_f2df5a8a9057783e,2026-03-31,2025-11-27,124,35980.0,35980.0,stale_visible,refresh,2
2,client_20259bd6705d81d4,content_ac4e2d9d3bbb06de,2026-03-31,2025-11-27,124,33348.0,33348.0,stale_visible,refresh,3
3,client_20259bd6705d81d4,content_66d1fffc91f4f029,2026-03-31,2025-11-27,124,31038.0,31038.0,stale_visible,refresh,4
4,client_20259bd6705d81d4,content_9598a57544925111,2026-03-31,2025-11-28,123,24066.0,24066.0,stale_visible,refresh,5
5,client_20259bd6705d81d4,content_b956947c822af734,2026-03-31,2025-11-27,124,22502.0,22502.0,stale_visible,refresh,6
6,client_20259bd6705d81d4,content_0d2aaf57d7146812,2026-03-31,2025-11-28,123,21060.0,21060.0,stale_visible,refresh,7
7,client_20259bd6705d81d4,content_b361694d518f80e2,2026-03-31,2025-11-27,124,18103.0,18103.0,stale_visible,refresh,8
8,client_20259bd6705d81d4,content_283e87bc4e224d58,2026-03-31,2025-11-27,124,16388.0,16388.0,stale_visible,refresh,9
9,client_20259bd6705d81d4,content_3f962469cb61a7c9,2026-03-31,2025-11-27,124,16019.0,16019.0,stale_visible,refresh,10


In [19]:
import os

os.makedirs("work/outputs", exist_ok=True)

queue.to_csv(
    "work/outputs/baseline_action_score.csv",
    index=False
)

print("Saved:", "work/outputs/baseline_action_score.csv")
print("Total rows:", len(queue))

Saved: work/outputs/baseline_action_score.csv
Total rows: 27886


In [18]:
display(
    queue[
        [
            "rank",
            "client_hash_id",
            "content_hash_id",
            "stale_days",
            "gsc_impressions",
            "score",
            "reason_code",
            "action"
        ]
    ].head(20)
)

,rank,client_hash_id,content_hash_id,stale_days,gsc_impressions,score,reason_code,action
0,1,client_20259bd6705d81d4,content_097459d155cccb26,124,37930.0,37930.0,stale_visible,refresh
1,2,client_20259bd6705d81d4,content_f2df5a8a9057783e,124,35980.0,35980.0,stale_visible,refresh
2,3,client_20259bd6705d81d4,content_ac4e2d9d3bbb06de,124,33348.0,33348.0,stale_visible,refresh
3,4,client_20259bd6705d81d4,content_66d1fffc91f4f029,124,31038.0,31038.0,stale_visible,refresh
4,5,client_20259bd6705d81d4,content_9598a57544925111,123,24066.0,24066.0,stale_visible,refresh
5,6,client_20259bd6705d81d4,content_b956947c822af734,124,22502.0,22502.0,stale_visible,refresh
6,7,client_20259bd6705d81d4,content_0d2aaf57d7146812,123,21060.0,21060.0,stale_visible,refresh
7,8,client_20259bd6705d81d4,content_b361694d518f80e2,124,18103.0,18103.0,stale_visible,refresh
8,9,client_20259bd6705d81d4,content_283e87bc4e224d58,124,16388.0,16388.0,stale_visible,refresh
9,10,client_20259bd6705d81d4,content_3f962469cb61a7c9,124,16019.0,16019.0,stale_visible,refresh


## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

## Top-20 review

| Rank | Action | Why it is here | What would make it wrong |
|---:|---|---|---|
| 1 | Refresh | 124 days stale and 37,930 March impressions, making it the highest-priority stale visible page. | The page may have intentionally stable content, or its high visibility may not be relevant to a refresh decision. |
| 2 | Refresh | 124 days stale with 35,980 March impressions, giving it very strong observed visibility. | The impressions may come from queries that are not appropriate targets for a content refresh. |
| 3 | Refresh | 124 days stale with 33,348 March impressions. | The page may still accurately satisfy its search intent despite being old. |
| 4 | Refresh | 124 days stale with 31,038 March impressions. | A refresh could change content that is already performing well and unintentionally hurt it. |
| 5 | Refresh | 123 days stale with 24,066 March impressions. | The page may not actually need substantive changes despite meeting the threshold. |
| 6 | Refresh | 124 days stale with 22,502 March impressions. | Its search visibility may be temporary or concentrated in a narrow query set. |
| 7 | Refresh | 123 days stale with 21,060 March impressions. | The content could be intentionally stable and still accurate. |
| 8 | Refresh | 124 days stale with 18,103 March impressions. | Updating it without understanding the reason for its visibility could reduce performance. |
| 9 | Refresh | 124 days stale with 16,388 March impressions. | The 90-day staleness threshold is only a rule-of-thumb and may not indicate actual content decay. |
| 10 | Refresh | 124 days stale with 16,019 March impressions. | The page may already contain the information users need and not benefit from rewriting. |
| 11 | Refresh | 124 days stale with 15,879 March impressions. | High impressions do not prove that refreshing the page will improve outcomes. |
| 12 | Refresh | 124 days stale with 14,774 March impressions. | The page may be ranking for stable evergreen information that does not require updating. |
| 13 | Refresh | 123 days stale with 10,024 March impressions. | It is close to the visibility threshold, so the priority may be overstated by the simple rule. |
| 14 | Refresh | 123 days stale with 9,961 March impressions. | Search impressions alone do not show whether the content has a quality or relevance problem. |
| 15 | Refresh | 123 days stale with 9,737 March impressions. | A refresh could be unnecessary if the underlying topic has not changed. |
| 16 | Refresh | 124 days stale with 9,177 March impressions. | The page may have stable performance despite its age. |
| 17 | Refresh | 124 days stale with 8,423 March impressions. | The page may not have a meaningful content deficiency that a refresh would solve. |
| 18 | Refresh | 123 days stale with 8,352 March impressions. | The observed impressions may not translate into an opportunity for improvement. |
| 19 | Refresh | 124 days stale with 7,544 March impressions. | The page may be correctly optimized already and simply have an older update date. |
| 20 | Refresh | 124 days stale with 6,853 March impressions. | It has lower visibility than the other selected pages, so the refresh priority may be weaker than the score suggests. |

### Skeptical review finding

A notable limitation is that all 20 highest-ranked pages belong to the same
client. This is not caused by using the client identifier as a feature; the
client identifier is used only for grouping and identification.

The concentration suggests that the baseline may prioritize high-volume
clients when their pages satisfy the same stale-and-visible rule. This is a
useful limitation to carry into the modeling phase rather than treating the
baseline ranking as a universal priority list across clients.

## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

## Weak picks

### Weak pick: Rank 20

The rank-20 page is a plausible but weak baseline recommendation. It satisfies
both hand-written thresholds (123 stale days and 6,853 impressions), so the
rule correctly selects it. However, the rule has no information about whether
the content is actually outdated, whether its search intent has changed, or
whether refreshing it would improve performance.

This illustrates the limitation of the baseline: staleness and visibility
identify a plausible refresh opportunity, but they do not establish that a
refresh will improve the page.

### Leakage check

The baseline uses only March 2026 observed data:

- `content_updated_date` from the content dimension
- March 2026 `gsc_impressions`
- March 2026 `report_date`

It does not use a future performance window, outcome label, trend label, or
future March/April performance.

Therefore the baseline is intended as a pre-model decision rule rather than a
rule fitted to a future outcome.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.